# 2/3 — Train (Kaggle GPU, checkpointed)

**Prerequisite:** run `01_dataset_prep_and_eda.ipynb` first, review its EDA output, then **Add Data** → attach that notebook's output version to *this* notebook before running anything below.

**Settings (right sidebar) → Accelerator → GPU T4 x2 (or P100).** GPU access needs phone-number verification on your Kaggle account (CPU-only without it).

This notebook is checkpoint/resume-aware — see Section 2 below — because a Kaggle session is capped (~9-12hr depending on source) and `/kaggle/working` does **not** survive a session restart unless you explicitly commit it first.

In [ ]:
!pip install -q ultralytics==8.4.106
import ultralytics
ultralytics.checks()

## 1. Locate the prepared dataset from notebook 1

Replace `<notebook-01-slug>` with whatever Kaggle named the attached input dataset (visible in the Data panel on the right) once you've added it.

In [ ]:
from pathlib import Path

INPUT_DATA_DIR = Path("/kaggle/input/<notebook-01-slug>")  # <-- EDIT: attached output of 01_dataset_prep_and_eda.ipynb
DATA_YAML = INPUT_DATA_DIR / "lostfound" / "lostfound.yaml"
assert DATA_YAML.exists(), f"{DATA_YAML} not found — check the input dataset is attached and the slug above is right."

## 2. Checkpoint / resume

Set `RESUME = True` only if you're continuing a previous attempt that got cut off. If so, **Add Data** → your own previous `02_train.ipynb` run's output version (the one you Saved right before the session ended) as a *second* input, then point `PREVIOUS_RUN_DIR` at it below. This copies that run's `weights/last.pt` + training state into place and resumes from the last completed epoch — it does **not** restart from `yolo26n.pt`.

In [ ]:
RESUME = False  # <-- set True to continue a previous cut-off attempt
PREVIOUS_RUN_DIR = Path("/kaggle/input/<notebook-02-previous-attempt-slug>/runs/lostfound_yolo26n")  # <-- EDIT if RESUME=True

RUN_PROJECT = "/kaggle/working/runs"
RUN_NAME = "lostfound_yolo26n"
RUN_DIR = Path(RUN_PROJECT) / RUN_NAME
LAST_CKPT = RUN_DIR / "weights" / "last.pt"

if RESUME:
    import shutil
    assert PREVIOUS_RUN_DIR.exists(), f"{PREVIOUS_RUN_DIR} not found — attach the previous run's output as an input dataset first."
    RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(PREVIOUS_RUN_DIR, RUN_DIR, dirs_exist_ok=True)
    if not LAST_CKPT.exists():
        raise SystemExit(
            f"RESUME=True but {LAST_CKPT} doesn't exist after copying {PREVIOUS_RUN_DIR}. "
            "The previous attempt likely never completed a single epoch (killed before its "
            "first checkpoint save), or the wrong output version is attached. Fix "
            "PREVIOUS_RUN_DIR above, or set RESUME=False to start fresh instead. Do NOT "
            "ignore this and re-run with RESUME=True anyway — the training cell below has "
            "no fallback, and would otherwise need to fail loudly here rather than silently "
            "starting a fresh run inside this now-partially-populated RUN_DIR, which would "
            "mix stale args.yaml/results.csv from the old attempt with the new run."
        )
    print(f"Copied previous run into {RUN_DIR}. Resuming from {LAST_CKPT}.")

## 3. Train

`freeze=10` — see `docs/FINE_TUNING_YOLO_GUIDE.md` Section 4 for the exact frozen/trainable parameter breakdown (1,115,744 frozen / 1,456,536 trainable, 56.62% of the model trains) and Section 5 for why this is full fine-tuning with a frozen backbone, not LoRA/QLoRA.

In [ ]:
from ultralytics import YOLO

# RESUME=True is only ever reached here once cell 5 above has already confirmed
# LAST_CKPT exists (it raises SystemExit otherwise) — no silent fresh-training
# fallback, so a missing checkpoint can never be mistaken for a real resume.
if RESUME:
    print(f"Resuming from {LAST_CKPT}")
    model = YOLO(str(LAST_CKPT))
    results = model.train(resume=True)
else:
    print("Starting a fresh training run.")
    model = YOLO("yolo26n.pt")
    results = model.train(
        data=str(DATA_YAML),
        epochs=100,
        imgsz=640,
        freeze=10,
        optimizer="AdamW",
        lr0=0.001,
        patience=15,
        project=RUN_PROJECT,
        name=RUN_NAME,
        exist_ok=True,
    )

## If the session is about to run out (Kaggle session-limit runbook)

Ultralytics writes `weights/last.pt` after every completed epoch, so progress isn't lost just because the clock is running low — but it only survives a session restart if you commit it:

1. **Interrupt the training cell** (Stop button) — don't let Kaggle kill it for you. Whatever epoch last finished is already safely on disk.
2. **File → Save Version** — commits `/kaggle/working` (including `runs/lostfound_yolo26n/`) as this notebook's Output, which becomes a Kaggle Dataset you can reattach.
3. Next session: open a fresh copy of this notebook, **Add Data** → that previous output version, set `RESUME = True` and `PREVIOUS_RUN_DIR` in Section 2 above, re-run from the top. `resume=True` continues from `last.pt`'s saved epoch + optimizer state, not from scratch.

## 4. Save the output for 03_evaluate_finetune_checks.ipynb

**File → Save Version** once training finishes (`patience=15` early-stops on plateau; expect roughly 1-3hr on a single T4 per `docs/FINE_TUNING_YOLO_GUIDE.md` Section 8). `03_evaluate_finetune_checks.ipynb` attaches this version's output (`runs/lostfound_yolo26n/weights/best.pt`) as an input.